# บทที่ 12 - การลดประวัติการแชทด้วย Agent Scratchpad

สมุดบันทึกนี้แสดงวิธีการจัดการบริบทในการสนทนาที่ยาวนานโดยใช้ Microsoft Agent Framework เมื่อการสนทนาเพิ่มขึ้น จำนวนโทเค็นก็เพิ่มตามไปด้วย — จนเกินขอบเขตหน้าต่างบริบทของโมเดล เราจะจัดการกับปัญหานี้ด้วย **รูปแบบการสรุปบริบท** และ **agent scratchpad** สำหรับหน่วยความจำที่คงอยู่

## สิ่งที่คุณจะได้เรียนรู้:
1. **ทำไมการจัดการบริบทจึงสำคัญ**: ความเข้าใจเกี่ยวกับขีดจำกัดโทเค็นและหน้าต่างบริบท
2. **Agent ที่รับรู้บริบท**: การสร้าง agent ที่จัดการบริบทในการสนทนาของตัวเอง
3. **รูปแบบการสรุปบริบท**: การใช้เครื่องมือเพื่อย่อยประวัติการสนทนา
4. **Agent Scratchpad**: หน่วยความจำถาวรที่คงอยู่แม้ลดบริบท

## ข้อกำหนดเบื้องต้น:
- การตั้งค่า Azure OpenAI พร้อมตัวแปรสภาพแวดล้อมที่กำหนดค่าไว้
- ความเข้าใจพื้นฐานเกี่ยวกับแนวคิด agent จากบทเรียนก่อนหน้า


## ตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ทำไมการจัดการบริบทถึงสำคัญ

ทุก LLM มี **หน้าต่างบริบท** ที่จำกัด — จำนวนโทเค็นสูงสุดที่สามารถประมวลผลได้ในคำขอเดียว เมื่อการสนทนาแบบหลายรอบดำเนินไป:

- **จำนวนโทเค็นเพิ่มขึ้นอย่างเส้นตรง** กับข้อความของผู้ใช้แต่ละข้อความและการตอบกลับของผู้ช่วย
- **โทเค็นคำใบ้อัตราเป็นต้นทุนหลัก** เพราะประวัติทั้งหมดจะถูกส่งใหม่ทุกครั้ง
- ในที่สุดการสนทนาจะ **เกินหน้าต่างบริบท** และโมเดลจะตัดทอนหรือเกิดข้อผิดพลาด

### กลยุทธ์ในการจัดการบริบท

| กลยุทธ์ | วิธีการทำงาน | ข้อแลกเปลี่ยน |
|---|---|---|
| **การตัดทอน** | ตัดข้อความที่เก่าที่สุดทิ้ง | สูญเสียบริบทเริ่มต้น |
| **การสรุป** | ย่อตัวข้อความเก่าเป็นบทสรุป | สูญเสียรายละเอียดบางส่วน แต่รักษาจุดสำคัญไว้ได้ |
| **สมุดจด / หน่วยความจำภายนอก** | เก็บข้อเท็จจริงสำคัญนอกการสนทนา | ต้องเรียกใช้เครื่องมือ แต่ทนต่อการลดขนาดได้ |

ในโน้ตบุ๊กนี้ เรารวม **การสรุป** กับ **เครื่องมือสมุดจด** เพื่อให้เอเย่นต์สามารถรักษาความต่อเนื่องแม้ประวัติการสนทนาถูกย่อให้กระชับขึ้น


## การสร้างเอเจนต์ที่มีความเข้าใจตามบริบท


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## การจำลองการสนทนาแบบยาว

มาลองผ่านการสนทนาหลายรอบเพื่อดูว่าบริบทจะสะสมอย่างไร ตัวแทนควรจดจำรายละเอียดสำคัญ (ความชอบ งบประมาณ วันที่เดินทาง) ข้ามรอบและแสดงความต่อเนื่องได้อย่างชัดเจน


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

สังเกตว่าเอเจนต์ยังคงรักษาบริบทจากรอบก่อนหน้า — มันรู้เกี่ยวกับญี่ปุ่น ซูชิ วัด การถ่ายภาพ งบประมาณ $3000 การเดินทางคนเดียว และช่วงเวลาเดือนเมษายน ในการสนทนาสั้น ๆ นี้ทำงานได้ดี แต่เมื่อบทสนทนายาวขึ้น ประวัติเต็มจะมีค่าใช้จ่ายสูงในการส่งซ้ำ

เรามาต่อบทสนทนาด้วยรอบเพิ่มเติมเพื่อดูการสะสมบริบท:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## รูปแบบการสรุปบริบท

เมื่อบทสนทนายาวขึ้น เราสามารถใช้ **เครื่องมือสรุป** เพื่อย่อบริบทที่สะสมมาให้อยู่ในรูปแบบกะทัดรัด ตัวแทนจะเรียกใช้เครื่องมือนี้เพื่อบันทึกความชื่นชอบสำคัญ เพื่อให้แม้ว่าข้อความเก่าจะถูกลบทิ้ง ข้อมูลสำคัญยังคงได้รับการเก็บรักษาไว้

รูปแบบนี้เป็นบล็อกสร้างสำหรับการลดประวัติที่ซับซ้อนมากขึ้น:
1. ตัวแทนระบุข้อเท็จจริงสำคัญจากบทสนทนา
2. เรียกใช้เครื่องมือสรุปเพื่อบันทึกข้อเท็จจริงเหล่านั้น
3. ข้อความเก่าสามารถถูกลบอย่างปลอดภัยเพราะสรุปจับใจความที่สำคัญ

ด้านล่างนี้เรากำหนดเครื่องมือ `summarize_preferences` ซึ่งตัวแทนสามารถเรียกใช้เพื่อบันทึกสรุปกะทัดรัดของสิ่งที่ได้เรียนรู้


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีจัดการบริบทในการสนทนาเอเจนต์ที่ดำเนินการเป็นเวลานานโดยใช้ Microsoft Agent Framework:

### แนวคิดหลัก
- **หน้าต่างบริบทมีขีดจำกัด** — ทุกโทเค็นในประวัติการสนทนาจะมีค่าใช้จ่ายและนับรวมกับขีดจำกัด
- **เครื่องมือสรุปความ** ช่วยให้เอเจนต์ย่อบริบทที่สะสมเป็นสรุปที่กระชับ ลดการใช้โทเค็นในขณะที่รักษาข้อมูลสำคัญไว้
- **สมุดจดเอเจนต์** ให้หน่วยความจำภายนอกที่อยู่ถาวรซึ่งไม่สูญหายเมื่อลดขนาดการสนทนา

### สิ่งที่คุณสร้าง
- **เอเจนต์ที่ตระหนักบริบท** ซึ่งรักษาความต่อเนื่องในการสนทนาแบบหลายรอบ
- **เครื่องมือสรุปความ** (`summarize_preferences`) ที่บันทึกรายละเอียดสำคัญของผู้ใช้ในรูปแบบที่กระชับ
- **การสนทนาแบบหลายรอบ** ที่แสดงการเก็บรักษาบริบทและการจัดการการเปลี่ยนแปลง

### การประยุกต์ใช้ในโลกจริง
- **บอทบริการลูกค้า**: จำค่ากำหนดที่ต้องการระหว่างเซสชันสนับสนุนที่ยาวนาน
- **ผู้ช่วยส่วนตัว**: ติดตามโครงการที่ดำเนินอยู่โดยไม่ต้องอธิบายบริบทซ้ำ
- **ครูติวเตอร์การศึกษา**: รักษาความก้าวหน้าของนักเรียนระหว่างการโต้ตอบหลายครั้ง

### ขั้นตอนถัดไป
- นำเครื่องมือสมุดจดถาวรแบบเต็มรูปแบบที่ใช้ไฟล์มาใช้งาน
- เพิ่มการตัดประวัติอัตโนมัติหลังการสรุปความ
- รวมกับฐานข้อมูลเวกเตอร์เพื่อค้นหาหน่วยความจำเชิงความหมาย
- สร้างเอเจนต์ที่สามารถกลับมาสนทนาใหม่ได้ในวันถัดไปพร้อมบริบทรอบด้านครบถ้วน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**: เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด แต่โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อน เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่น่าเชื่อถือ สำหรับข้อมูลสำคัญแนะนำให้ใช้การแปลโดยผู้เชี่ยวชาญมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใดๆ ที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
